# 04 Sequence Language Model

## 목적
`x`와 `y`가 한 칸 shifted된 시퀀스임을 확인하고, token embedding과 positional embedding을 더한 뒤 `(B,T,V)` logits로 모든 위치의 다음 문자를 예측합니다. 이 단계에는 attention이 없습니다.

In [1]:
from pathlib import Path
import sys, torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.tokenizer import CharTokenizer, load_text
from src.dataset import CharSequenceDataset
from src.baselines import SequenceNoAttention

torch.manual_seed(42)
text = load_text(str(ROOT / 'data/korean_finance_corpus.txt'))
tokenizer = CharTokenizer(text)
ids = tokenizer.encode(text)

## 핵심 코드
`CharSequenceDataset`은 `block_size+1` 토큰 조각에서 `x=chunk[:-1]`, `y=chunk[1:]`를 만듭니다. 모델은 token embedding과 positional embedding을 더하지만 attention 없이 위치별 MLP만 적용합니다.

In [2]:
block_size = 8
ds = CharSequenceDataset(ids[:160], block_size=block_size)
x0, y0 = ds[0]
print('x text:', tokenizer.decode(x0.tolist()))
print('y text:', tokenizer.decode(y0.tolist()))
x = torch.stack([ds[i][0] for i in range(4)])
y = torch.stack([ds[i][1] for i in range(4)])
print('x shape:', tuple(x.shape), 'targets shape:', tuple(y.shape))

x text: 인공지능은 금융
y text: 공지능은 금융 
x shape: (4, 8) targets shape: (4, 8)


In [3]:
model = SequenceNoAttention(tokenizer.vocab_size, block_size=block_size, n_embd=24)
logits, loss = model(x, y)
print('logits shape (B,T,V):', tuple(logits.shape))
print('targets shape (B,T):', tuple(y.shape))
print('sequence cross entropy:', round(float(loss), 4))

logits shape (B,T,V): (4, 8, 370)
targets shape (B,T): (4, 8)
sequence cross entropy: 6.0738


/tmp/ipykernel_68755/1567170053.py:5: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  print('sequence cross entropy:', round(float(loss), 4))


## 이 단계의 한계
위치 임베딩은 순서 정보를 주지만, 각 위치가 이전 위치의 내용을 섞어 읽는 attention은 아직 없습니다.

## 다음 단계가 해결하는 점
다음 노트북은 Q, K, V와 causal mask를 사용해 각 위치가 과거 위치 정보를 가중 평균으로 읽는 masked self-attention을 만듭니다.